# Game Playtime Analytics Dashboard — corrected Colab notebook

Run the code cells **from top to bottom**. The ETL and Streamlit app both use exactly the same database path: `output/backlog.db`, and the same table: `curated_games`.

This notebook uses your GitHub dataset `hltb_dataset_normalized.zip` and keeps the existing game-playtime dashboard concept.


In [ ]:
# Cell 1 — Install Python dependencies
!pip -q install --upgrade pandas streamlit plotly


In [ ]:
# Cell 2 — Download/inspect the source dataset
import pandas as pd

CSV_URL = "https://raw.githubusercontent.com/blacklack547-hash/game-playtime-dashboard/main/hltb_dataset_normalized.zip"

source_df = pd.read_csv(CSV_URL)
print("Dataset loaded successfully.")
print("Rows:", len(source_df))
print("Columns:", source_df.columns.tolist())
display(source_df.head())


In [ ]:
# Cell 3 — Create the ETL pipeline script
%%writefile etl_pipeline.py
from pathlib import Path
import sqlite3
import pandas as pd

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / "output"
DB_PATH = OUTPUT_DIR / "backlog.db"
CSV_URL = "https://raw.githubusercontent.com/blacklack547-hash/game-playtime-dashboard/main/hltb_dataset_normalized.zip"

def run_etl():
    print("=== STARTING HLTB PLAYTIME ETL PIPELINE ===")

    # EXTRACT
    df = pd.read_csv(CSV_URL)
    print(f"Extracted {len(df):,} rows from the GitHub dataset.")

    # TRANSFORM
    df.columns = (
        df.columns.astype(str)
        .str.replace(" ", "_", regex=False)
        .str.lower()
        .str.strip()
    )

    before = len(df)
    if "name" in df.columns:
        df = df.dropna(subset=["name"])
    df = df.drop_duplicates()
    print(f"Removed {before - len(df):,} duplicate/invalid rows.")

    playtime_cols = ["main_story", "main_plus_sides", "completionist", "all_styles"]
    for col in playtime_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
            median_value = df[col].median()
            if pd.isna(median_value):
                median_value = 0
            df[col] = df[col].fillna(median_value)

    text_cols = ["name", "type", "platform", "genres", "release_date", "source_url"]
    for col in text_cols:
        if col in df.columns:
            df[col] = df[col].fillna("").astype(str).str.strip()

    # LOAD — this is the one canonical database path used by app.py
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    with sqlite3.connect(DB_PATH) as conn:
        df.to_sql("curated_games", conn, if_exists="replace", index=False)

        # Verify the write before declaring success
        count = pd.read_sql_query(
            "SELECT COUNT(*) AS n FROM curated_games", conn
        ).iloc[0]["n"]

    print(f"Loaded {count:,} records into: {DB_PATH}")
    print("ETL completed successfully.")
    return count

if __name__ == "__main__":
    run_etl()


In [ ]:
# Cell 4 — Execute the ETL pipeline
!python etl_pipeline.py


In [ ]:
# Cell 5 — Verify the exact database file and table used by the dashboard
from pathlib import Path
import sqlite3
import pandas as pd

DB_PATH = Path.cwd() / "output" / "backlog.db"

print("Working directory:", Path.cwd())
print("Database path:", DB_PATH)
print("Database exists:", DB_PATH.exists())
print("Database size (bytes):", DB_PATH.stat().st_size if DB_PATH.exists() else "MISSING")

if not DB_PATH.exists():
    raise FileNotFoundError(f"ETL database not found: {DB_PATH}")

with sqlite3.connect(DB_PATH) as conn:
    tables = pd.read_sql_query(
        "SELECT name FROM sqlite_master WHERE type='table'", conn
    )
    print("\nTables:")
    display(tables)

    if "curated_games" not in tables["name"].tolist():
        raise RuntimeError("The curated_games table was not created. Re-run Cell 4.")

    count = pd.read_sql_query(
        "SELECT COUNT(*) AS total FROM curated_games", conn
    ).iloc[0]["total"]
    print(f"\nRows in curated_games: {count:,}")
    display(pd.read_sql_query("SELECT * FROM curated_games LIMIT 5", conn))

if count == 0:
    raise RuntimeError("curated_games exists but contains zero rows.")


In [ ]:
# Cell 6 — Write the Streamlit dashboard
%%writefile app.py
from pathlib import Path
import sqlite3
import pandas as pd
import streamlit as st
import plotly.express as px

st.set_page_config(
    page_title="Game Playtime Analytics Dashboard",
    page_icon="🎮",
    layout="wide"
)

# IMPORTANT: Must match the ETL output path exactly.
DB_PATH = Path(__file__).resolve().parent / "output" / "backlog.db"
TABLE_NAME = "curated_games"

@st.cache_data
def load_optimized_data(db_path: str, modified_time: float):
    # modified_time is included so Streamlit cache refreshes after ETL reruns.
    with sqlite3.connect(db_path) as conn:
        return pd.read_sql_query(f"SELECT * FROM {TABLE_NAME}", conn)

st.title("🎮 Video Game Playtime Curation & Analytics Platform")
st.caption("Explore HowLongToBeat-style playtime estimates across the game catalog.")

if not DB_PATH.exists():
    st.error(f"Database not found: {DB_PATH}")
    st.info("Run the ETL pipeline cell first, then restart this dashboard.")
    st.stop()

if DB_PATH.stat().st_size == 0:
    st.error(f"The database file exists but is empty: {DB_PATH}")
    st.info("Re-run the ETL pipeline cell and verify its success message.")
    st.stop()

try:
    df = load_optimized_data(str(DB_PATH), DB_PATH.stat().st_mtime)
except Exception as exc:
    st.error(f"Could not read '{TABLE_NAME}' from the database.")
    st.exception(exc)
    st.stop()

if df.empty:
    st.error("The curated_games table contains zero rows.")
    st.info("Run the ETL pipeline again and check Cell 5.")
    st.stop()

# Ensure numeric playtime columns are numeric even if source values vary.
playtime_cols = ["main_story", "main_plus_sides", "completionist", "all_styles"]
for col in playtime_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

st.success(f"Loaded {len(df):,} games from {DB_PATH}")

# KPI cards
k1, k2, k3 = st.columns(3)
k1.metric("🎮 Total catalogued titles", f"{len(df):,}")

if "main_story" in df.columns:
    valid_main = df.loc[df["main_story"] > 0, "main_story"]
    avg_main = valid_main.mean() if not valid_main.empty else 0
    k2.metric("⏱️ Average main story", f"{avg_main:.1f} hours")
else:
    k2.metric("⏱️ Average main story", "N/A")

if "completionist" in df.columns:
    valid_comp = df.loc[df["completionist"] > 0, "completionist"]
    avg_comp = valid_comp.mean() if not valid_comp.empty else 0
    k3.metric("🏆 Average 100% run", f"{avg_comp:.1f} hours")
else:
    k3.metric("🏆 Average 100% run", "N/A")

st.divider()

# Sidebar filters
st.sidebar.header("🔎 Filter games")
filtered = df.copy()

if "name" in filtered.columns:
    search_text = st.sidebar.text_input("Search game title")
    if search_text:
        filtered = filtered[
            filtered["name"].astype(str).str.contains(search_text, case=False, na=False)
        ]

if "type" in filtered.columns:
    type_values = sorted(filtered["type"].dropna().astype(str).unique().tolist())
    selected_types = st.sidebar.multiselect("Game type", type_values, default=type_values)
    if selected_types:
        filtered = filtered[filtered["type"].astype(str).isin(selected_types)]

if "platform" in filtered.columns:
    platform_values = sorted(filtered["platform"].dropna().astype(str).unique().tolist())
    selected_platforms = st.sidebar.multiselect("Platform", platform_values)
    if selected_platforms:
        filtered = filtered[filtered["platform"].astype(str).isin(selected_platforms)]

st.subheader("📈 Playtime insights")
left, right = st.columns(2)

with left:
    if {"name", "completionist"}.issubset(filtered.columns):
        longest = filtered[filtered["completionist"] > 0].nlargest(10, "completionist")
        st.markdown("#### 🏆 Top 10 longest games (100% completion)")
        if not longest.empty:
            fig = px.bar(
                longest.sort_values("completionist"),
                x="completionist", y="name", orientation="h",
                labels={"completionist": "Hours", "name": "Game"}
            )
            st.plotly_chart(fig, use_container_width=True)
        else:
            st.info("No completionist playtime values available for this selection.")

with right:
    if {"name", "main_story"}.issubset(filtered.columns):
        shortest = filtered[filtered["main_story"] > 0].nsmallest(10, "main_story")
        st.markdown("#### ⚡ 10 shortest main stories")
        if not shortest.empty:
            fig = px.bar(
                shortest.sort_values("main_story", ascending=False),
                x="main_story", y="name", orientation="h",
                labels={"main_story": "Hours", "name": "Game"}
            )
            st.plotly_chart(fig, use_container_width=True)
        else:
            st.info("No main-story playtime values available for this selection.")

st.divider()
st.subheader("🗂️ Browse game catalog")

display_cols = [
    c for c in [
        "name", "type", "platform", "main_story",
        "main_plus_sides", "completionist", "all_styles", "release_date"
    ] if c in filtered.columns
]

if display_cols:
    st.caption(f"Showing {len(filtered):,} matching games.")
    st.dataframe(
        filtered[display_cols].head(500),
        use_container_width=True,
        hide_index=True
    )
else:
    st.warning("Expected game columns were not found. Available columns:")
    st.write(df.columns.tolist())

st.caption("The table above is limited to the first 500 matching rows for responsiveness.")


In [ ]:
# Cell 7 — Install Cloudflare Tunnel (one-time per runtime)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared


In [ ]:
# Cell 8 — Launch Streamlit and expose it with Cloudflare Tunnel
# Run this cell once. It will keep running while the dashboard is open.
!streamlit run app.py --server.port 8501 --server.headless true & ./cloudflared tunnel --url http://localhost:8501
